# HAVC bước 1: chẩn đoán head đọc chữ tiếng Anh vs tiếng Việt (Qwen2.5-VL-3B)

Pilot cho C4/C5. Không dùng dataset QA dev/test.
Với từng ảnh chữ, đưa văn bản đúng làm câu trả lời (teacher forcing), tại mỗi token mô hình đọc đúng ghi vị trí
attention cao nhất của từng head (36 layer x 16 head) và xem nó có rơi vào từ đang đọc không.

**Cần:** dataset Kaggle `da2-havc` (`src/`, `data/diagnostic/corpus_*.jsonl`, `data/diagnostic/fonts/`), tạo bằng
`python -m src.data_prep.pack_kaggle --name da2-havc --datasets --diagnostic`; GPU T4 x2, Internet On (tải model).

**Chạy lại 17/11/2026:** từ ngày này vision tower chạy fp32 (`upcast_vision`, xem `src/vlm/qwen.py`) vì vision block
cuối tràn số ở fp16. Danh sách expert head của C4/C5 phải chọn lại với cấu hình này; cần upload `da2-havc` bản mới. Ảnh được vẽ lại trên Kaggle từ văn bản + font đã chốt (seed cố định).

**Thời gian ước tính:** vẽ ảnh vài phút; chẩn đoán khoảng 0,5-1 s/ảnh, 2.000 ảnh khoảng 20-40 phút.
Muốn xem nhanh trước thì đặt `N_IMAGES = 300` ở ô cấu hình.

**Kết quả:** tải `/kaggle/working/havc_diagnosis_results.zip` về, giải nén vào gốc repo (thư mục `results/`).

In [ ]:
!pip install -q -U transformers accelerate
import os, sys, shutil, glob
hits = glob.glob('/kaggle/input/**/src/diagnosis/head_scores.py', recursive=True)
assert hits, 'Không thấy src/diagnosis/head_scores.py: kiểm tra đã Add Input dataset da2-havc chưa'
src_root = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')
import torch, transformers
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0), '| transformers', transformers.__version__)

In [ ]:
N_IMAGES = 1000  # mỗi ngôn ngữ; 300 để chạy thử nhanh
!python -X utf8 -m src.diagnosis.render_text --n {N_IMAGES}

In [ ]:
import json, random, time
import numpy as np
from pathlib import Path
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from src.vlm.qwen import upcast_vision
from src.diagnosis.head_scores import collect, PROMPT
from src.diagnosis.analyze_heads import summarize, load, expert_heads, K, THRESH

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CFG = dict(model='Qwen/Qwen2.5-VL-3B-Instruct', dtype='float16', vision_dtype='float32', attn='eager', max_pixels=1024 * 28 * 28,
           prompt=PROMPT, n_images=N_IMAGES, seed=SEED, K=K, thresh=THRESH,
           method='HAVC buoc 1 (OCR head scoring), teacher forcing, moi layer x moi head; xem src/diagnosis/head_scores.py',
           # pip install -U nên phiên bản có thể đổi giữa các lần chạy: ghi lại để tái lập
           torch_version=torch.__version__, transformers_version=transformers.__version__,
           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
processor = AutoProcessor.from_pretrained(CFG['model'], max_pixels=CFG['max_pixels'])
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG['model'], dtype=torch.float16, device_map='auto', attn_implementation='eager').eval()
upcast_vision(model)  # vision tower fp32, LLM fp16: tránh tràn số fp16 ở vision block cuối (src/vlm/qwen.py)
print('layers x heads:', model.config.text_config.num_hidden_layers, 'x', model.config.text_config.num_attention_heads)

In [ ]:
OUT = {'en': Path('results/C4_havc_en'), 'vi': Path('results/C5_havc_vi')}
stats = {}
for lang in ('en', 'vi'):
    rows = [json.loads(l) for l in open(f'data/diagnostic/{lang}/meta.jsonl', encoding='utf-8')][:N_IMAGES]
    OUT[lang].mkdir(parents=True, exist_ok=True)
    print(lang, len(rows), 'ảnh')
    for i in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(i)
    t0 = time.time()
    stats[lang] = collect(model, processor, rows, 'data/diagnostic', OUT[lang] / f'diagnosis_{lang}.npz')
    sec = time.time() - t0
    # VRAM đỉnh từng GPU (model chia trên 2 T4 với device_map='auto')
    peak = [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]
    cost = {'seconds_total': round(sec, 1), 'seconds_per_image': round(sec / max(1, len(rows)), 3), 'peak_vram_gb': peak}
    print(lang, stats[lang], cost)
    json.dump(CFG | {'lang': lang, 'stats': stats[lang], 'cost': cost},
              open(OUT[lang] / f'diagnosis_{lang}_config.json', 'w'), ensure_ascii=False, indent=1)
    torch.cuda.empty_cache()

In [ ]:
D = {lang: load(OUT[lang] / f'diagnosis_{lang}.npz') for lang in ('en', 'vi')}
res = summarize(D['en'], D['vi'])
json.dump(res, open(OUT['vi'] / 'diagnosis_summary.json', 'w'), ensure_ascii=False, indent=1)
for lang in ('en', 'vi'):
    r = res[lang]
    print(f"{lang}: đọc đúng {r['read_acc_token']:.1%} token | head > {THRESH}: {r['n_heads_above_thresh']} | "
          f"tỉ lệ trúng tốt nhất {r['best_hit_rate']:.1%} (ngẫu nhiên {r['chance_hit_rate']:.1%})")
    print('   top-K:', [(h['layer'], h['head'], h['hit_rate']) for h in r['top_k']])
print('Trùng nhau:', res['overlap'])
print(json.dumps(res['vi_heldout'], ensure_ascii=False, indent=1))

# Expert head (đầu vào của C4/C5 lúc inference), giống `analyze_heads --export-heads`
for lang in ('en', 'vi'):
    eh = expert_heads(D[lang], f'diagnosis_{lang}.npz')
    json.dump(eh, open(OUT[lang] / 'expert_heads_config.json', 'w'), indent=1)
    print(lang, 'expert head:', eh['n_heads'])

In [ ]:
!cd /kaggle/working/da2 && zip -qr /kaggle/working/havc_diagnosis_results.zip results/C4_havc_en results/C5_havc_vi
!ls -la /kaggle/working/havc_diagnosis_results.zip